## <로스엔젤레스 박물관 데이터 수집>
- 스크래핑을 통한 수집 (JSON API)

### 1. 필요한 것들 불러오기

In [1]:
import os
import time
import requests
import pandas as pd

MUSEUM_CODE = "LACMA"
OUTPUT_EXCEL = f"../data/{MUSEUM_CODE}_hyungbae.xlsx"
HEADERS = {
    "User-Agent": "aks-digital-humanities-research/1.0",
    "Content-Type": "application/json",
}

### 2. API 호출해서 원하는 정보 얻기

In [2]:
CATEGORY = "흉배"
CATEGORY_LETTER = "H"
KEYWORD = "rank badge"

rows = []
serial_counter = {}

def next_temp_id(museum_code):
    n = serial_counter.get(museum_code, 0) + 1
    serial_counter[museum_code] = n
    return f"Y{museum_code}{CATEGORY_LETTER}{n:02d}"

def fetch_lacma(keyword=KEYWORD, per_page=48):
    url = "https://collections.lacma.org/api/search"
    page = 1
    all_items = []

    while True:
        payload = {
            "query": keyword,
            "classification": [], "department": [], "artist": [], "placeMade": [],
            "creditLine": [], "culture": [], "period": [], "style": [],
            "building": [], "gallery": [],
            "onView": False, "hasImage": True, "publicDomain": False,
            "sort": "RELEVANCE", "page": page, "perPage": per_page,
        }
        resp = requests.post(url, headers=HEADERS, json=payload, timeout=20)
        resp.raise_for_status()
        data = resp.json()
        results = data.get("results") or []
        all_items.extend(results)

        total = data.get("total", 0)
        if page * per_page >= total or not results:
            break
        page += 1
        time.sleep(0.3)

    print(f"전체 검색 결과: {len(all_items)}건 (필터링 전)")

    for item in all_items:
        obj = (item.get("data") or {}).get("object") or {}

        titles = obj.get("titles") or []
        # 영어명: Primary Title 중 displayOrder가 가장 앞선 것
        primary_titles = [t for t in titles if t.get("titleType") == "Primary Title"]
        primary_titles.sort(key=lambda t: t.get("displayOrder", 999))
        title_en = primary_titles[0]["title"] if primary_titles else (titles[0]["title"] if titles else None)

        # 한글명: 번역 제목이 있으면 사용
        ko_titles = [t for t in titles if "translation" in (t.get("titleType") or "").lower()]
        title_ko = ko_titles[0]["title"] if ko_titles else ""

        classification = obj.get("classification")
        # Costumes 분류가 아니면 흉배가 아닐 가능성이 높음 -> 필터링
        if classification != "Costumes":
            continue
        # 제목에 흉배 관련 단어가 없으면 제외
        title_check = (title_en or "").lower()
        if not any(w in title_check for w in ["badge", "buzi", "hyungbae"]):
            continue

        temp_id = next_temp_id(MUSEUM_CODE)

        images = obj.get("images") or []
        image_urls = [img.get("renditions", {}).get("desktop") for img in images if img.get("renditions", {}).get("desktop")]

        constituents = obj.get("constituents") or []
        artists = "; ".join(c.get("displayName", "") for c in constituents if c.get("displayName"))

        place_made = obj.get("placeMade") or []
        place_str = "; ".join(place_made) if isinstance(place_made, list) else place_made

        rows.append({
            "임시ID": temp_id,
            "분류": CATEGORY,
            "소장처": "로스앤젤레스 카운티 미술관(LACMA)",
            "소장처유물번호": obj.get("accessionNumber"),
            "한글명": title_ko,
            "한자명": "",
            "영어명": title_en,
            "URL": f"https://collections.lacma.org/object/{item.get('id')}",
            "searched_keyword": keyword,
            "classification": classification,
            "department": obj.get("department"),
            "artist": artists,
            "date": obj.get("dated"),
            "medium": obj.get("medium"),
            "dimensions": obj.get("dimensions"),
            "credit_line": obj.get("creditLine"),
            "place_made": place_str,
            "culture": obj.get("culture"),
            "image_url": image_urls[0] if image_urls else None,
            "image_urls": "; ".join(image_urls),
            "license_note": "© Museum Associates/LACMA (건별 라이선스 확인 필요)",
        })

fetch_lacma()
print(f"필터링 후 흉배 대상: {len(rows)}건")

전체 검색 결과: 274건 (필터링 전)
필터링 후 흉배 대상: 14건


### 3. 데이터 프레임 -> 엑셀

In [3]:
df = pd.DataFrame(rows)
master_cols = ["임시ID", "분류", "소장처", "소장처유물번호", "한글명", "한자명", "영어명",
               "URL", "searched_keyword", "classification", "department", "artist", "date",
               "medium", "dimensions", "credit_line", "place_made", "culture",
               "image_url", "image_urls", "license_note"]
other_cols = [c for c in df.columns if c not in master_cols]
df = df[master_cols + other_cols]

os.makedirs("../data", exist_ok=True)
df.to_excel(OUTPUT_EXCEL, index=False)
print(df.shape)
df.head()

(14, 21)


,임시ID,분류,소장처,소장처유물번호,한글명,한자명,영어명,URL,searched_keyword,classification,...,artist,date,medium,dimensions,credit_line,place_made,culture,image_url,image_urls,license_note
0,YLACMAH01,흉배,로스앤젤레스 카운티 미술관(LACMA),M.79.86,,,Senatorial Rank Badge (Stole),https://collections.lacma.org/object/37332,rank badge,Costumes,...,Unknown,circa 1500,"Silk, pile-on-pile velvet weave",65 1/2 x 8 1/2 in. (166.37 x 21.59 cm),Gift of Mr. and Mrs. Dennis C. Stanfill,"Italy, Venice",None,https://collections-images.lacma.org/images/37...,https://collections-images.lacma.org/images/37...,© Museum Associates/LACMA (건별 라이선스 확인 필요)
1,YLACMAH02,흉배,로스앤젤레스 카운티 미술관(LACMA),M.39.2.375,,,Badge (Buzi) of the Third Civil Rank with Two ...,https://collections.lacma.org/object/1165,rank badge,Costumes,...,Unknown,"Ming dynasty (1368-1644), 16th century",Silk and metallic-wrapped thread tapestry weav...,14 x 14 in. (35.56 x 35.56 cm),Gift of Miss Bella Mabury,China,None,https://collections-images.lacma.org/images/11...,https://collections-images.lacma.org/images/11...,© Museum Associates/LACMA (건별 라이선스 확인 필요)
2,YLACMAH03,흉배,로스앤젤레스 카운티 미술관(LACMA),M.2000.15.196a,기린흉배,,Rank Badge (Hyungbae) with Mythical Animal (Gi...,https://collections.lacma.org/object/120730,rank badge,Costumes,...,Unknown,"Joseon dynasty, 1392-1910, 1864-1892",Silk damask with metallic-thread embroidery,9 3/4 x 8 5/8 in. (24.77 x 21.91 cm),Purchased with Museum Funds,Korea,None,https://collections-images.lacma.org/images/12...,https://collections-images.lacma.org/images/12...,© Museum Associates/LACMA (건별 라이선스 확인 필요)
3,YLACMAH04,흉배,로스앤젤레스 카운티 미술관(LACMA),M.2000.15.197b,금사쌍학흉배,,Rank Badge (Hyungbae) of Civil Official with T...,https://collections.lacma.org/object/120700,rank badge,Costumes,...,Unknown,"Joseon dynasty (1392-1910), 19th century",Silk and metallic thread embroidery on silk da...,7 1/4 x 8 in. (18.42 x 20.32 cm),Purchased with Museum Funds,Korea,None,https://collections-images.lacma.org/images/12...,https://collections-images.lacma.org/images/12...,© Museum Associates/LACMA (건별 라이선스 확인 필요)
4,YLACMAH05,흉배,로스앤젤레스 카운티 미술관(LACMA),M.39.2.234,四品武官補子 \t\t清代晚期\t刺繡\r\n,,Rank Badge (Buzi) of the Fourth Military Rank ...,https://collections.lacma.org/object/1123,rank badge,Costumes,...,Unknown,"Qing dynasty (1644-1912), early 19th century",Silk satin with silk and metallic-wrapped thre...,12 1/4 × 12 5/8 in. (31.12 × 32.07 cm),Gift of Miss Bella Mabury,China,None,https://collections-images.lacma.org/images/11...,https://collections-images.lacma.org/images/11...,© Museum Associates/LACMA (건별 라이선스 확인 필요)


### 4. 이미지

In [4]:
# 이미지 저장 폴더 (NFM 노트북이랑 동일한 경로 구조)
os.makedirs("../image/hyungbae", exist_ok=True)

pure_hyungbae = df.reset_index(drop=True)

for i, row in pure_hyungbae.iterrows():
    temp_id = row["임시ID"]
    raw = row["image_urls"]

    if pd.isna(raw) or str(raw).strip() == "":
        continue  # 이미지 없는 유물은 건너뜀

    image_urls = [u for u in str(raw).split("; ") if u.strip()]

    for idx, image_url in enumerate(image_urls):
        suffix = "" if len(image_urls) == 1 else f"-{idx + 1}"
        filepath = f"../image/hyungbae/{temp_id}{suffix}.jpg"

        resp = requests.get(image_url, headers=HEADERS, timeout=30)
        with open(filepath, "wb") as f:
            f.write(resp.content)

    if (i + 1) % 20 == 0:
        print(f"{i + 1} / {len(pure_hyungbae)} 완료")

    time.sleep(0.5)